# Reto Salud y Bienestar — Recolección de datos abiertos (datos.gov.co)

Empezamos de cero con las fuentes exactas que encontraste:

1. **Morbilidad General** (`gbte-byrg`)
2. **Cobertura de Vacunación PAI en el Valle del Cauca** (`uw8e-gzpp`)
3. **Calidad del Aire en Colombia — Promedio Anual** (`kekd-7v7h`)
4. **DIVIPOLA** (`gdxc-w37w`) — para normalizar nombres de municipio/departamento

**Enfoque:** en el intento anterior (precipitación) pedimos agregaciones pesadas del lado del servidor
y eso causó timeouts. Esta vez el notebook primero **explora** cada dataset (columnas, tamaño real,
muestra de filas) y solo después decide cómo descargarlo: todo de una vez si es chico, o paginado sin
agregación si es grande. Así no repetimos el mismo error a ciegas.


In [ ]:
!pip install sodapy pandas -q

In [ ]:
import time
import pandas as pd
from sodapy import Socrata

client = Socrata("www.datos.gov.co", None, timeout=90)

AÑOS = ["2018", "2019", "2021"]

DATASETS = {
    "divipola": "gdxc-w37w",
    "morbilidad": "gbte-byrg",
    "vacunacion_valle": "uw8e-gzpp",
    "calidad_aire": "kekd-7v7h",
}
print("Cliente Socrata listo.")

## Paso 1 — Explorar cada dataset antes de descargar nada

Para cada uno: columnas reales, una muestra de 3 filas, y un conteo total de filas (`count(*)`, una
consulta liviana que no calcula nada complejo, solo cuenta). Esto nos dice si podemos traerlo completo
de un solo golpe o si necesitamos paginar.

In [ ]:
def explorar(nombre, dataset_id):
    print(f"\n=== {nombre} ({dataset_id}) ===")
    try:
        muestra = client.get(dataset_id, limit=3)
        df_muestra = pd.DataFrame.from_records(muestra)
        print("Columnas:", list(df_muestra.columns))
        display(df_muestra)
    except Exception as e:
        print(f" No se pudo traer una muestra: {e}")
        return None

    try:
        conteo = client.get(dataset_id, select="count(*)")
        total = int(conteo[0]["count"])
        print(f"Total de filas: {total:,}")
    except Exception as e:
        print(f" No se pudo contar filas ({e}) — asumimos que puede ser grande.")
        total = None

    return {"columnas": list(df_muestra.columns), "total_filas": total}

info_datasets = {}
for nombre, dataset_id in DATASETS.items():
    info_datasets[nombre] = explorar(nombre, dataset_id)
    time.sleep(0.5)

## Paso 2 — Descarga adaptativa

Reglas simples según lo que vimos arriba:
- Si el dataset tiene **menos de 200.000 filas**, lo traemos completo de un solo golpe.
- Si es más grande, lo traemos **paginado** (sin pedirle ningún cálculo/agregación a Socrata) y filtrando
  por año si logramos detectar una columna de fecha/año — así cada página es liviana.

La detección de la columna de año es automática por palabras clave; si falla, avisa en vez de fallar en silencio.

In [ ]:
def encontrar_columna(columnas, palabras_clave):
    for col in columnas:
        for palabra in palabras_clave:
            if palabra in col.lower():
                return col
    return None

def descargar(nombre, dataset_id, umbral=200000, tamano_pagina=50000, pausa=0.5):
    info = info_datasets.get(nombre)
    if info is None:
        print(f" {nombre}: no hay info de exploración, me la salto. Revisa el Paso 1.")
        return None

    columnas = info["columnas"]
    total = info["total_filas"]
    col_anio = encontrar_columna(columnas, ["anio", "año", "ano", "vigencia", "year", "fecha"])

    filas = []
    if total is not None and total <= umbral:
        print(f"{nombre}: {total:,} filas — se trae completo de un golpe.")
        filas = client.get(dataset_id, limit=total + 1000)
    else:
        print(f"{nombre}: dataset grande o de tamaño desconocido — paginando"
              + (f" filtrado por columna '{col_anio}'." if col_anio else " sin filtro de año detectado."))
        offset = 0
        while True:
            try:
                kwargs = dict(limit=tamano_pagina, offset=offset)
                if col_anio:
                    where_clause = " OR ".join([f"{col_anio} like '%{a}%'" for a in AÑOS])
                    kwargs["where"] = where_clause
                pagina = client.get(dataset_id, **kwargs)
            except Exception as e:
                print(f" Falló la página offset={offset}: {e}")
                break
            if not pagina:
                break
            filas.extend(pagina)
            print(f"  offset {offset}: +{len(pagina)} filas (acumulado {len(filas)})")
            offset += tamano_pagina
            time.sleep(pausa)
            if len(pagina) < tamano_pagina:
                break

    if not filas:
        print(f" {nombre}: no se trajo ninguna fila.")
        return None

    df = pd.DataFrame.from_records(filas)
    archivo = f"{nombre}.csv"
    df.to_csv(archivo, index=False)
    print(f"Guardado {archivo} con {len(df)} filas")
    return df

df_divipola = descargar("divipola", DATASETS["divipola"])

In [ ]:
df_morbilidad = descargar("morbilidad", DATASETS["morbilidad"])

In [ ]:
df_vacunacion_valle = descargar("vacunacion_valle", DATASETS["vacunacion_valle"])

In [ ]:
df_calidad_aire = descargar("calidad_aire", DATASETS["calidad_aire"])

In [ ]:
try:
    from google.colab import files
    for f in ["divipola.csv", "morbilidad.csv", "vacunacion_valle.csv", "calidad_aire.csv"]:
        try:
            files.download(f)
        except FileNotFoundError:
            print(f"{f} no se generó — revisa los mensajes de la celda correspondiente arriba.")
except ImportError:
    print("No estás en Colab — los archivos ya quedaron guardados en la carpeta donde corriste este notebook.")